The best solution has fitness -4.0, and its Adam = B, Ben = D, Chloe = B, Daniel = B, Ethan = C, Felix = A, Grace = A, Hannah = B, Isaac = A, James = A, Katy = C, Leo = A, Mike = A, Noah = A, Olivia = B, Peter = A, Quinn = B, Riley = A
For task 1
With modification [('Isaac', 'B'), ('Olivia', 'A')], the fitness is -4.0
With modification [('Isaac', 'B')], the fitness is -12.0
With modification [('Adam', 'A')], the fitness is -20.0
With modification [('Adam', 'A'), ('Isaac', 'B')], the fitness is -28.0



The best solution has fitness -4.0, and its Alice = C, Brandon = C, Clara = C, Dominic = B, Eleanor = B, Fiona = D, Gabriel = C, Hazel = B, Ivy = A, Joshua = A, Kevin = D, Lucas = C, Matilda = B, Nathan = B, Oscar = A, Phoebe = D, Quentin = D, Rose = A

For task 2
With modification [('Brandon', 'D'), ('Kevin', 'C')], the fitness is -4.0
With modification [('Alice', 'D'), ('Phoebe', 'C')], the fitness is -12.0
With modification [('Lucas', 'D'), ('Phoebe', 'C')], the fitness is -12.0
With modification [('Alice', 'D'), ('Lucas', 'D')], the fitness is -20.0
With modification [('Alice', 'D'), ('Brandon', 'D')], the fitness is -20.0
With modification [('Alice', 'D'), ('Kevin', 'C')], the fitness is -20.0
With modification [('Lucas', 'D'), ('Brandon', 'D')], the fitness is -20.0
With modification [('Lucas', 'D'), ('Kevin', 'C')], the fitness is -20.0
With modification [('Phoebe', 'C'), ('Brandon', 'D')], the fitness is -28.0
With modification [('Phoebe', 'C'), ('Kevin', 'C')], the fitness is -28.0

In [5]:
task_1_answers = {  'Put Isaac on B and Olivia on A': 4,
                    'Put Isaac on B': 12,
                    'Put Adam on A': 20,
                    'Put Adam on A and Isaac on B': 28}

task_2_answers = {'Brandon to D;Kevin to C':4,
                  'Alice to D;Phoebe to C':12,
                  'Lucas to D;Phoebe to C':12,
                  'Alice to D;Lucas to D':20,
                  'Alice to D;Brandon to D':20,
                  'Alice to D;Kevin to C':20,
                  'Brandon to D;Lucas to D':20,
                  'Kevin to C;Lucas to D':20,
                  'Brandon to D;Phoebe to C':28,
                  'Kevin to C;Phoebe to C':28,
}

def get_scores_for_task_1(task_1_answer: str): 
    items = task_1_answer.split(";")
    assert(all(item in task_1_answers for item in items))
    return [task_1_answer[item] for item in items]

def get_score_for_task_2(task_2_answer: str):
    actual_key = task_2_answer
    if actual_key not in task_2_answers:
        actual_key = ";".join(reversed(actual_key.split(";")))
    if actual_key not in task_2_answers:
        raise Exception(f"The key {task_2_answer} is not recognised")
    
    return task_2_answers[actual_key]

In [14]:
import pandas as pd

def get_task_data():
    data_path = r"C:\Users\gac8\PycharmProjects\PS-descriptors-LCS\UserStudy\data\answers.csv"
    
    raw_df = pd.read_csv(data_path)
    
    df = raw_df[["Which form?", "Task 1 arrangement", "Task 2 tickboxes"]]
    return df.rename(columns={"Which form?":"Group", "Task 1 arrangement":"Task 1", "Task 2 tickboxes": "Task 2"})
    

display(get_task_data())
    

,Group,Task 1,Task 2
0,A,Put Adam on A and Isaac on B;Put Isaac on B an...,Kevin to C;Brandon to D
1,B,Put Isaac on B and Olivia on A;Put Isaac on B;...,Brandon to D;Kevin to C


# Code Description
* Distance Calculation:
  The code computes the distance between each participant's ranking and the ground truth using the Spearman footrule metric. This metric sums the absolute differences in positions for each item in the ranking compared to its position in the ground truth.

* Statistical Testing:
  Two statistical tests are carried out on the computed distances:

    * Independent Samples t-test:
    This test compares the mean distances between the two groups (X and Y) assuming normally distributed data.
    * Mann-Whitney U Test:
    A non-parametric alternative that does not assume normality, used to assess if one group tends to have lower distances (i.e., is closer to the ground truth) than the other.


In [15]:
import utils
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu

# Define the ground truth ranking and create a lookup for item positions.
ground_truth = list(task_1_answers.items())
ground_truth.sort(key=utils.second)
ground_truth = [item for item, value in ground_truth]
ground_truth_index = {item: idx for idx, item in enumerate(ground_truth)}

print("The ground truth is")
for option, ranking in ground_truth_index.items():
    print(f"{option = }, {ranking = }")

def compute_distance(ranking_str, ground_truth_index):
    """
    Computes the Spearman footrule distance between a given ranking (as a semicolon-separated string)
    and the ground truth. It sums the absolute differences in positions for each item.
    """
    # Split the string into a list of items.
    ranking = ranking_str.split(";")
    distance = 0
    for i, item in enumerate(ranking):
        if item in ground_truth_index:
            distance += abs(i - ground_truth_index[item])
        else:
            print(f"Warning: {item} not in ground truth.")
    return distance

df = get_task_data()

# Compute the distance for each participant.
df['Distance'] = df['Task 1'].apply(lambda x: compute_distance(x, ground_truth_index))
print("Data with computed distances:")
print(df)

# Separate the distances by group.
group_a = df[df['Group'] == "A"]['Distance']
group_b = df[df['Group'] == "B"]['Distance']

# Perform an independent samples t-test.
t_stat, p_val = ttest_ind(group_a, group_b, equal_var=False)
print("\nT-test results:")
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.3f}")

# Perform a Mann-Whitney U test.
u_stat, p_val_mw = mannwhitneyu(group_a, group_b, alternative='two-sided')
print("\nMann-Whitney U test results:")
print(f"U statistic = {u_stat:.3f}, p-value = {p_val_mw:.3f}")


The ground truth is
option = 'Put Isaac on B and Olivia on A', ranking = 0
option = 'Put Isaac on B', ranking = 1
option = 'Put Adam on A', ranking = 2
option = 'Put Adam on A and Isaac on B', ranking = 3
Data with computed distances:
  Group                                             Task 1  \
0     A  Put Adam on A and Isaac on B;Put Isaac on B an...   
1     B  Put Isaac on B and Olivia on A;Put Isaac on B;...   

                    Task 2  Distance  
0  Kevin to C;Brandon to D         6  
1  Brandon to D;Kevin to C         2  

T-test results:
t-statistic = nan, p-value = nan

Mann-Whitney U test results:
U statistic = 1.000, p-value = 1.000


C:\Users\gac8\PycharmProjects\PS-descriptors-LCS\venv\lib\site-packages\scipy\stats\_stats_py.py:1113: RuntimeWarning: divide by zero encountered in divide
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
C:\Users\gac8\PycharmProjects\PS-descriptors-LCS\venv\lib\site-packages\scipy\stats\_stats_py.py:1113: RuntimeWarning: invalid value encountered in scalar multiply
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero


# Code Description
* Score Mapping:
  The code maps each user's answer in the "Task 2" column to a corresponding score using the task_2_answers dictionary.

* Group Comparison:
  The resulting scores are separated by the "Group" column (with groups "X" and "Y").

* Statistical Testing:
  Two tests are performed to compare the scores between the groups:

    * Independent Samples t-test:
      This test compares the mean scores of the two groups.
    * Mann-Whitney U Test:
      A non-parametric test that assesses whether the distributions of scores differ significantly between the groups.

In [18]:
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu

df = get_task_data().copy()

# Map each answer to its corresponding score.
df['Score'] = df['Task 2'].map(get_score_for_task_2)

display(df)

# Split the scores by group.
group_a_scores = df[df['Group'] == 'A']['Score']
group_b_scores = df[df['Group'] == 'B']['Score']

# Perform an independent samples t-test.
t_stat, p_val = ttest_ind(group_a_scores, group_b_scores, equal_var=False)
print("T-test results:")
print(f"t-statistic: {t_stat:.3f}, p-value: {p_val:.3f}")

# Perform a Mann-Whitney U test.
u_stat, p_val_mw = mannwhitneyu(group_a_scores, group_b_scores, alternative='two-sided')
print("\nMann-Whitney U test results:")
print(f"U statistic: {u_stat:.3f}, p-value: {p_val_mw:.3f}")


,Group,Task 1,Task 2,Score
0,A,Put Adam on A and Isaac on B;Put Isaac on B an...,Kevin to C;Brandon to D,4
1,B,Put Isaac on B and Olivia on A;Put Isaac on B;...,Brandon to D;Kevin to C,4


T-test results:
t-statistic: nan, p-value: nan

Mann-Whitney U test results:
U statistic: 0.500, p-value: 1.000


C:\Users\gac8\PycharmProjects\PS-descriptors-LCS\venv\lib\site-packages\scipy\stats\_stats_py.py:1113: RuntimeWarning: divide by zero encountered in divide
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
C:\Users\gac8\PycharmProjects\PS-descriptors-LCS\venv\lib\site-packages\scipy\stats\_stats_py.py:1113: RuntimeWarning: invalid value encountered in scalar multiply
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
